<span style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">An Exception was encountered at '<a href="#papermill-error-cell">In [15]</a>'.</span>

In [1]:
# Parameters
model_dir = "model31.0"


In [2]:
import pandas as pd, numpy as np
from pathlib import Path
from vivarium import Artifact
import db_queries
from get_draws.api import get_draws

from vivarium_testing_utils.automated_validation import ValidationContext

from vivarium_gates_mncnh.validation.measures import NeonatalCauseSpecificMortalityRisk, NeonatalOtherCausesMortalityRisk, NeonatalPretermBirthMortalityRisk

In [3]:
import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

In [4]:
results_dir = Path('/mnt/team/simulation_science/pub/models/vivarium_gates_mncnh/results/automated_vv/ethiopia/2025_12_19_14_51_22/')

In [5]:
# Create validation context
vc = ValidationContext(results_dir)

In [6]:
vc.age_groups

,,
age_start,age_end,age_group
0.000000,0.019178,Early Neonatal
0.019178,0.076712,Late Neonatal
0.076712,0.500000,1-5 months
0.500000,1.000000,6-11 months
1.000000,2.000000,12 to 23 months
2.000000,5.000000,2 to 4
5.000000,10.000000,5 to 9
10.000000,15.000000,10 to 14
15.000000,20.000000,15 to 19


In [7]:
# List outputs
vc.get_sim_outputs()

['abortion_miscarriage_ectopic_pregnancy_counts',
 'abortion_miscarriage_ectopic_pregnancy_ylds',
 'all_causes_relative_risk',
 'anc_hemoglobin',
 'anc_other',
 'antibiotics',
 'azithromycin',
 'births',
 'cpap',
 'maternal_disorders_burden_observer_disorder_deaths',
 'maternal_disorders_burden_observer_disorder_ylls',
 'maternal_hemorrhage_counts',
 'maternal_hemorrhage_ylds',
 'maternal_obstructed_labor_and_uterine_rupture_counts',
 'maternal_obstructed_labor_and_uterine_rupture_ylds',
 'maternal_sepsis_and_other_maternal_infections_counts',
 'maternal_sepsis_and_other_maternal_infections_ylds',
 'misoprostol',
 'neonatal_acmrisk_count',
 'neonatal_acmrisk_nonzero_count',
 'neonatal_acmrisk_sum',
 'neonatal_acmrisk_sum_of_squares',
 'neonatal_burden_observer_disorder_deaths',
 'neonatal_burden_observer_disorder_ylls',
 'neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma_death_counts',
 'neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma_relative_risk',
 'neonatal_impossib

In [8]:
# Artifact keys
vc.get_artifact_keys()

['cause.abortion_miscarriage_ectopic_pregnancy.cause_specific_mortality_rate',
 'cause.abortion_miscarriage_ectopic_pregnancy.incidence_rate',
 'cause.abortion_miscarriage_ectopic_pregnancy.yld_rate',
 'cause.all_causes.adjusted_birth_counts',
 'cause.all_causes.all_cause_mortality_risk',
 'cause.all_causes.cause_specific_mortality_rate',
 'cause.ectopic_pregnancy.raw_incidence_rate',
 'cause.facility_choice.bemonc_facility_fraction',
 'cause.facility_choice.probability_bemonc_birth',
 'cause.facility_choice.probability_cemonc_birth',
 'cause.facility_choice.probability_home_birth',
 'cause.facility_choice.probability_home_birth_given_full_term',
 'cause.facility_choice.probability_home_birth_given_preterm',
 'cause.maternal_abortion_and_miscarriage.raw_incidence_rate',
 'cause.maternal_hemorrhage.cause_specific_mortality_rate',
 'cause.maternal_hemorrhage.incidence_rate',
 'cause.maternal_hemorrhage.yld_rate',
 'cause.maternal_obstructed_labor_and_uterine_rupture.cause_specific_mortal

In [9]:
# Subset to neonatal mortality risk artifact keys
[key for key in vc.get_artifact_keys() if "mortality_risk" in key]

['cause.all_causes.all_cause_mortality_risk',
 'cause.neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma.mortality_risk',
 'cause.neonatal_preterm_birth.mortality_risk',
 'cause.neonatal_sepsis_and_other_neonatal_infections.mortality_risk']

In [10]:
# Simulation outputs
vc.get_sim_outputs()

['abortion_miscarriage_ectopic_pregnancy_counts',
 'abortion_miscarriage_ectopic_pregnancy_ylds',
 'all_causes_relative_risk',
 'anc_hemoglobin',
 'anc_other',
 'antibiotics',
 'azithromycin',
 'births',
 'cpap',
 'maternal_disorders_burden_observer_disorder_deaths',
 'maternal_disorders_burden_observer_disorder_ylls',
 'maternal_hemorrhage_counts',
 'maternal_hemorrhage_ylds',
 'maternal_obstructed_labor_and_uterine_rupture_counts',
 'maternal_obstructed_labor_and_uterine_rupture_ylds',
 'maternal_sepsis_and_other_maternal_infections_counts',
 'maternal_sepsis_and_other_maternal_infections_ylds',
 'misoprostol',
 'neonatal_acmrisk_count',
 'neonatal_acmrisk_nonzero_count',
 'neonatal_acmrisk_sum',
 'neonatal_acmrisk_sum_of_squares',
 'neonatal_burden_observer_disorder_deaths',
 'neonatal_burden_observer_disorder_ylls',
 'neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma_death_counts',
 'neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma_relative_risk',
 'neonatal_impossib

In [11]:
[o for o in vc.get_sim_outputs() if 'death_counts' in o]

['neonatal_encephalopathy_due_to_birth_asphyxia_and_trauma_death_counts',
 'neonatal_preterm_birth_with_rds_death_counts',
 'neonatal_preterm_birth_without_rds_death_counts',
 'neonatal_sepsis_and_other_neonatal_infections_death_counts',
 'other_causes_death_counts']

In [12]:
sepsis_compare_key = "cause.neonatal_sepsis_and_other_neonatal_infections.mortality_risk"

In [13]:
# Add custom NeonatalCauseSpecificMortalityRisk measure class measure mapper
# This allows the ValidationContext to use custom measure classes that are not 
# included in the standard measure classes in VTU
vc.add_new_measure(sepsis_compare_key, NeonatalCauseSpecificMortalityRisk)

In [14]:
# Compare simulation outputs to artifact
vc.add_comparison(
    sepsis_compare_key,
    test_source="sim",
    ref_source="artifact",
)

2026-04-02 18:09:29.329 | INFO     | vivarium_testing_utils.automated_validation.data_transformation.age_groups:_format_dataframe:587 - Rebinning DataFrame age groups from <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe5504a990> to <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe551c6310>.


2026-04-02 18:09:32.526 | INFO     | vivarium_testing_utils.automated_validation.data_transformation.age_groups:_format_dataframe:587 - Rebinning DataFrame age groups from <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe55079050> to <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe5508b190>.


2026-04-02 18:09:35.322 | INFO     | vivarium_testing_utils.automated_validation.data_transformation.age_groups:_format_dataframe:587 - Rebinning DataFrame age groups from <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe54f9a410> to <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe54f9b3d0>.


2026-04-02 18:09:35.583 | INFO     | vivarium_testing_utils.automated_validation.data_transformation.age_groups:_format_dataframe:587 - Rebinning DataFrame age groups from <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe55007310> to <vivarium_testing_utils.automated_validation.data_transformation.age_groups.AgeSchema object at 0x7fbe54f66410>.


<span id="papermill-error-cell" style="color:red; font-family:Helvetica Neue, Helvetica, Arial, sans-serif; font-size:2em;">Execution using papermill encountered an exception here and stopped:</span>

In [15]:
# Comparison metadata
sepsis_metadata = vc.metadata(sepsis_compare_key)

TypeError: ValidationContext.metadata() missing 2 required positional arguments: 'test_source' and 'ref_source'

In [ ]:
sepsis_csmrisk_frame = vc.get_frame(
    sepsis_compare_key, 
    # aggregate_draws=True, 
    # stratifications=[],
)
sepsis_csmrisk_frame

In [ ]:
sepsis_csmrisk_frame.index.names

In [ ]:
sepsis_csmrisk_neonates = vc.get_frame(
    sepsis_compare_key, 
    aggregate_draws=True, 
    # stratifications=[],
    filters={"age_group": ["Early Neonatal", "Late Neonatal"]}
)
sepsis_csmrisk_neonates

In [ ]:
vc.plot_comparison(
    sepsis_compare_key, type="line", x_axis="sex", condition={"age_group": ['Early Neonatal', 'Late Neonatal']}, subplots=True
)

In [ ]:
# Need to get GBD data to cache custom GBD data since we cannot not pull mortality ratio from vivarium_inputs 
# Cause ids:
# all_causes: 294
# Preterm birth: 381
# Neonatal encephalopathy due to birth asphyxia : 382
# neonatal sepsis: 383
# 
gbd_births = db_queries.get_population(location_id=[179],
                                release_id=16,
                                year_id=2023,
                                age_group_id=164,
                                sex_id=[1, 2])
gbd_deaths = get_draws(location_id=[179],
                                release_id=16,
                                year_id=2023,
                                source='codcorrect',
                                metric_id=1,
                                measure_id=1,
                                gbd_id=[383],
                                age_group_id=[2,3], # enn and lnn
                                gbd_id_type='cause_id')

In [ ]:
draw_cols = [col for col in gbd_deaths.columns if "draw" in col]

In [ ]:
gbd_enn_0 = gbd_births.set_index(['location_id','sex_id'])[['population']]
gbd_enn_deaths = gbd_deaths.loc[gbd_deaths.age_group_id==2]
gbd_enn_mortality_ratio = gbd_enn_deaths.merge(gbd_enn_0, on=['location_id','sex_id'])
for col in draw_cols:
    gbd_enn_mortality_ratio[col] = gbd_enn_mortality_ratio[col] / gbd_enn_mortality_ratio.population
gbd_enn_mortality_ratio = gbd_enn_mortality_ratio.drop(columns='population').reset_index(drop=True)

gbd_enn_mortality_ratio

In [ ]:
# Population of LNN age group start
gbd_lnn_0 = gbd_enn_deaths.merge(gbd_enn_0, on=['location_id','sex_id'])
for col in draw_cols:
    gbd_lnn_0[col] = gbd_lnn_0.population - gbd_lnn_0[col]
gbd_lnn_0 = gbd_lnn_0.drop(columns='population')
gbd_lnn_0["age_group_id"] = 3  # This comes from gbd_enn_deaths above
gbd_lnn_0 = gbd_lnn_0.set_index([col for col in gbd_lnn_0.columns if col not in draw_cols])
gbd_lnn_0

In [ ]:
gbd_lnn_deaths = gbd_deaths.loc[gbd_deaths.age_group_id==3]
gbd_lnn_deaths = gbd_lnn_deaths.set_index([col for col in gbd_lnn_deaths.columns if col not in draw_cols])
gbd_lnn_deaths

In [ ]:
gbd_lnn_mortality_ratio = (gbd_lnn_deaths / gbd_lnn_0).reset_index()
gbd_lnn_mortality_ratio

In [ ]:
gbd_mortality_ratio = pd.concat([gbd_enn_mortality_ratio, gbd_lnn_mortality_ratio])
gbd_mortality_ratio

In [ ]:
# Cache custom GBD data so we can then use that to compare the simulation to GBD
vc.cache_gbd_data(sepsis_compare_key, gbd_mortality_ratio)

In [ ]:
# Cache adjust birth counts for weight aggregation
gbd_enn_0 = gbd_enn_0.reset_index()
gbd_enn_0["age_group_id"] = 2
gbd_enn_0

In [ ]:
enn_draw_pop = pd.DataFrame(
    {draw: gbd_enn_0['population'] for draw in draw_cols}
).reset_index(drop=True)
# enn_draw_pop["age_group_id"] = 2
# enn_draw_pop["year_id"] = 2023
enn_draw_pop

In [ ]:
enn_pop = pd.concat([gbd_enn_0, enn_draw_pop], axis=1)
enn_pop['year_id'] = 2023
enn_pop = enn_pop.drop(columns='population')
enn_pop

In [ ]:
gbd_lnn_0 = gbd_lnn_0.reset_index()
gbd_lnn_0 = gbd_lnn_0[["location_id", "sex_id", "age_group_id", "year_id"] + draw_cols]
gbd_lnn_0

In [ ]:
sepsis_adjusted_birth_counts = pd.concat([enn_pop, gbd_lnn_0])
sepsis_adjusted_birth_counts

In [ ]:
# Note: I understand this is very hidden that the weights for GBD data also have to be cached.
# What is a good way to communicate this?
vc.cache_gbd_data("cause.neonatal_sepsis_and_other_neonatal_infections.adjusted_birth_counts", sepsis_adjusted_birth_counts)

In [ ]:
# Now let's compare the simulation to GBD
# Note: This currently overwrites the previous comparison that had the same key.
# This is will be updated at some point so users will have access to all comparisons
vc.add_comparison(
    sepsis_compare_key,
    test_source="sim",
    ref_source="gbd",
)

In [ ]:
sepsis_sim_gbd = vc.get_frame(
    sepsis_compare_key, 
    aggregate_draws=True, 
    # stratifications=[],
    filters={"age_group": ["Early Neonatal", "Late Neonatal"]}
)
sepsis_sim_gbd

In [ ]:
vc.plot_comparison(
    sepsis_compare_key, type="line", x_axis="sex", condition={"age_group": ['Early Neonatal', 'Late Neonatal']}, subplots=True
)

In [ ]:
preterm_birth_key = "cause.neonatal_preterm_birth.mortality_risk"

In [ ]:
vc.add_new_measure(preterm_birth_key, NeonatalPretermBirthMortalityRisk)

In [ ]:
vc.add_comparison(
    preterm_birth_key,
    test_source="sim",
    ref_source="artifact",
)

In [ ]:
vc.metadata(preterm_birth_key)

In [ ]:
preterm_diff = vc.get_frame(
    preterm_birth_key, 
    aggregate_draws=True, 
    # stratifications=[],
    filters={"age_group": ["Early Neonatal", "Late Neonatal"]}
)
preterm_diff

In [ ]:
other_causes_key = "cause.other_causes.mortality_risk"

In [ ]:
vc.add_new_measure(other_causes_key, NeonatalOtherCausesMortalityRisk)

In [ ]:
vc.add_comparison(
    other_causes_key,
    "sim",
    "artifact",
)

In [ ]:
other_causes_diff = vc.get_frame(
    other_causes_key, 
    aggregate_draws=True, 
    # stratifications=[],
    filters={"age_group": ["Early Neonatal", "Late Neonatal"]}
)
other_causes_diff